# 📥 Data Fetching from Materials Project

**s-CGCNN Version 0.1 - Notebook 1**

This notebook demonstrates how to fetch crystal structure data from Materials Project API for GaAs and AlAs.

---

## Setup

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from src.data_acquisition.mp_fetcher import MPDataFetcher
from src.utils.logger_config import setup_logger

print("✓ Imports successful")

## 1. Initialize MPDataFetcher

Read API key from config file and initialize fetcher.

In [ ]:
# Read API key
api_key_file = Path("../config/mp_api_key.txt")

if not api_key_file.exists():
    print("❌ ERROR: API key not found!")
    print("Please create config/mp_api_key.txt with your Materials Project API key")
else:
    with open(api_key_file, 'r') as f:
        api_key = f.read().strip()
    print("✓ API key loaded")
    print(f"  Key length: {len(api_key)} characters")

In [ ]:
# Initialize fetcher
fetcher = MPDataFetcher(api_key, output_dir="../data/raw")
print("✓ MPDataFetcher initialized")
print(f"  Materials to fetch: {list(fetcher.materials.keys())}")

## 2. Check for Existing Data

Before fetching, check if data already exists locally.

In [ ]:
# Check for existing data
gaas_data = fetcher.load_saved_data("GaAs")
alas_data = fetcher.load_saved_data("AlAs")

if gaas_data and alas_data:
    print("✓ Data already exists!")
    print(f"  GaAs: {gaas_data['structure'].composition}")
    print(f"  AlAs: {alas_data['structure'].composition}")
    data_exists = True
else:
    print("⚠ Data not found. Will fetch from Materials Project.")
    data_exists = False

## 3. Fetch Data from Materials Project

**Note:** This step takes 30-60 seconds. Skip if data already exists.

In [ ]:
if not data_exists:
    print("Fetching data from Materials Project...")
    print("This may take 30-60 seconds...\n")
    
    all_data = fetcher.fetch_all_materials()
    
    print(f"\n✓ Fetched {len(all_data)} materials")
    
    # Reload data
    gaas_data = fetcher.load_saved_data("GaAs")
    alas_data = fetcher.load_saved_data("AlAs")
else:
    print("✓ Using existing data")

## 4. Explore GaAs Data

In [ ]:
print("="*60)
print("GaAs (mp-2534) Properties")
print("="*60)

# Structure
print(f"\nStructure:")
print(f"  Composition: {gaas_data['structure'].composition}")
print(f"  Formula: {gaas_data['structure'].composition.reduced_formula}")
print(f"  Lattice: a = {gaas_data['structure'].lattice.a:.4f} Å")
print(f"  Space group: {gaas_data['structure'].get_space_group_info()[0]}")
print(f"  Number of sites: {len(gaas_data['structure'])}")

# Properties
props = gaas_data['properties']
print(f"\nElectronic Properties:")
print(f"  Band gap: {props['band_gap']:.3f} eV")
print(f"  Gap type: {'Direct' if props['is_gap_direct'] else 'Indirect'}")
print(f"  Fermi energy: {props['efermi']:.3f} eV")

print(f"\nPhysical Properties:")
print(f"  Density: {props['density']:.3f} g/cm³")
print(f"  Formation energy: {props['formation_energy_per_atom']:.3f} eV/atom")

if 'elastic_tensor' in props:
    print(f"\nMechanical Properties:")
    print(f"  Bulk modulus: {props['elastic_tensor']['bulk_modulus_vrh']:.1f} GPa")
    print(f"  Shear modulus: {props['elastic_tensor']['shear_modulus_vrh']:.1f} GPa")

## 5. Explore AlAs Data

In [ ]:
print("="*60)
print("AlAs (mp-2172) Properties")
print("="*60)

# Structure
print(f"\nStructure:")
print(f"  Composition: {alas_data['structure'].composition}")
print(f"  Formula: {alas_data['structure'].composition.reduced_formula}")
print(f"  Lattice: a = {alas_data['structure'].lattice.a:.4f} Å")
print(f"  Space group: {alas_data['structure'].get_space_group_info()[0]}")
print(f"  Number of sites: {len(alas_data['structure'])}")

# Properties
props = alas_data['properties']
print(f"\nElectronic Properties:")
print(f"  Band gap: {props['band_gap']:.3f} eV")
print(f"  Gap type: {'Direct' if props['is_gap_direct'] else 'Indirect'}")
print(f"  Fermi energy: {props['efermi']:.3f} eV")

print(f"\nPhysical Properties:")
print(f"  Density: {props['density']:.3f} g/cm³")
print(f"  Formation energy: {props['formation_energy_per_atom']:.3f} eV/atom")

if 'elastic_tensor' in props:
    print(f"\nMechanical Properties:")
    print(f"  Bulk modulus: {props['elastic_tensor']['bulk_modulus_vrh']:.1f} GPa")
    print(f"  Shear modulus: {props['elastic_tensor']['shear_modulus_vrh']:.1f} GPa")

## 6. Compare GaAs and AlAs

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    'Property': [
        'Lattice constant (Å)',
        'Band gap (eV)',
        'Gap type',
        'Density (g/cm³)',
        'Formation energy (eV/atom)'
    ],
    'GaAs': [
        f"{gaas_data['structure'].lattice.a:.4f}",
        f"{gaas_data['properties']['band_gap']:.3f}",
        'Direct' if gaas_data['properties']['is_gap_direct'] else 'Indirect',
        f"{gaas_data['properties']['density']:.3f}",
        f"{gaas_data['properties']['formation_energy_per_atom']:.3f}"
    ],
    'AlAs': [
        f"{alas_data['structure'].lattice.a:.4f}",
        f"{alas_data['properties']['band_gap']:.3f}",
        'Direct' if alas_data['properties']['is_gap_direct'] else 'Indirect',
        f"{alas_data['properties']['density']:.3f}",
        f"{alas_data['properties']['formation_energy_per_atom']:.3f}"
    ]
})

print("\nGaAs vs AlAs Comparison:")
print(comparison.to_string(index=False))

## 7. Visualize Crystal Structures

Display both structures side-by-side.

In [ ]:
# Simple text representation
print("GaAs Structure:")
print(gaas_data['structure'])

print("\n" + "="*60 + "\n")

print("AlAs Structure:")
print(alas_data['structure'])

## 8. Summary Statistics

In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)
print(f"\n✓ Successfully fetched data for 2 materials")
print(f"  • GaAs (mp-2534): {gaas_data['structure'].composition}")
print(f"  • AlAs (mp-2172): {alas_data['structure'].composition}")
print(f"\n✓ Data saved to: data/raw/")
print(f"  • mp_2534_GaAs.json")
print(f"  • mp_2172_AlAs.json")
print(f"\n✓ Ready for structure interpolation!")
print(f"\n→ Next: Run notebook 02_Structure_Interpolation_Demo.ipynb")

---

## ✅ Checklist

- [x] API key loaded
- [x] GaAs data fetched
- [x] AlAs data fetched
- [x] Properties extracted
- [x] Data saved locally

**Status:** Complete ✓